In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", "Engineering", 75000, 25),
    (2, "Bob", "Marketing", 60000, 30),
    (3, "Charlie", "Engineering", 80000, 35),
    (4, "David", "Sales", 65000, 28),
    (5, "Eve", "Marketing", 70000, 32),
    (6, "Frank", "Engineering", 85000, 29),
    (7, "Grace", "Sales", 68000, 31),
    (8, "Hannah", "Marketing", 72000, 27),
    (9, "Ian", "Engineering", 90000, 40),
    (10, "Jack", "HR", 58000, 26),
    (11, "Karen", "Finance", 76000, 34),
    (12, "Leo", "Engineering", 82000, 33),
    (13, "Mia", "Sales", 67000, 29),
    (14, "Nathan", "Marketing", 71000, 36),
    (15, "Olivia", "HR", 62000, 28),
    (16, "Paul", "Finance", 78000, 41),
    (17, "Quinn", "Engineering", 87000, 38),
    (18, "Rachel", "Sales", 69000, 30),
    (19, "Steve", "Marketing", 74000, 35),
    (20, "Tina", "Engineering", 92000, 42)
]
columns = ["Id", "Name", "Department", "Salary", "Age"]
df = spark.createDataFrame(data, columns);

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 04:20:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 04:20:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
print("before partition:", df.rdd.getNumPartitions())

before partition: 4


In [3]:
df_repartitioned = df.repartition(10)

In [4]:
print("after partion:", df_repartitioned.rdd.getNumPartitions())

[Stage 0:>                                                          (0 + 4) / 4]

after partion: 10


In [6]:
df_coalesced = df_repartitioned.coalesce(2)

In [8]:
print("after coalesced:",df_coalesced.rdd.getNumPartitions())

after coalesced: 2


In [9]:
df_select.show()

NameError: name 'df_select' is not defined

In [10]:
df.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  2|    Bob|  Marketing| 60000| 30|
|  3|Charlie|Engineering| 80000| 35|
|  4|  David|      Sales| 65000| 28|
|  5|    Eve|  Marketing| 70000| 32|
|  6|  Frank|Engineering| 85000| 29|
|  7|  Grace|      Sales| 68000| 31|
|  8| Hannah|  Marketing| 72000| 27|
|  9|    Ian|Engineering| 90000| 40|
| 10|   Jack|         HR| 58000| 26|
| 11|  Karen|    Finance| 76000| 34|
| 12|    Leo|Engineering| 82000| 33|
| 13|    Mia|      Sales| 67000| 29|
| 14| Nathan|  Marketing| 71000| 36|
| 15| Olivia|         HR| 62000| 28|
| 16|   Paul|    Finance| 78000| 41|
| 17|  Quinn|Engineering| 87000| 38|
| 18| Rachel|      Sales| 69000| 30|
| 19|  Steve|  Marketing| 74000| 35|
| 20|   Tina|Engineering| 92000| 42|
+---+-------+-----------+------+---+



In [11]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.appName("EmployeeDF").getOrCreate()

departments = ["Engineering", "Marketing", "Sales", "HR", "Finance"]

data = []

for i in range(1, 101):
    data.append(
        (
            i,                          # EmployeeID
            f"Employee_{i}",            # Name
            random.choice(departments), # Department
            random.randint(50000, 100000), # Salary
            random.randint(22, 45)      # Age
        )
    )

columns = ["EmployeeID", "Name", "Department", "Salary", "Age"]

df = spark.createDataFrame(data, columns)

df.show(100, truncate=False)

26/06/13 04:51:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+----------+------------+-----------+------+---+
|EmployeeID|Name        |Department |Salary|Age|
+----------+------------+-----------+------+---+
|1         |Employee_1  |Sales      |61114 |34 |
|2         |Employee_2  |HR         |57506 |42 |
|3         |Employee_3  |HR         |90231 |41 |
|4         |Employee_4  |HR         |95529 |26 |
|5         |Employee_5  |HR         |99654 |30 |
|6         |Employee_6  |Finance    |84902 |43 |
|7         |Employee_7  |HR         |76794 |39 |
|8         |Employee_8  |Engineering|99773 |30 |
|9         |Employee_9  |Sales      |71572 |34 |
|10        |Employee_10 |Finance    |67648 |36 |
|11        |Employee_11 |HR         |51242 |32 |
|12        |Employee_12 |Marketing  |90634 |39 |
|13        |Employee_13 |HR         |53252 |38 |
|14        |Employee_14 |Sales      |71868 |31 |
|15        |Employee_15 |Finance    |50784 |43 |
|16        |Employee_16 |Marketing  |96940 |28 |
|17        |Employee_17 |Engineering|95902 |34 |
|18        |Employee

In [12]:
print(df.rdd.getNumPartitions())  

4


In [13]:
df_repartitioned = df.repartition(8)

print(df_repartitioned.rdd.getNumPartitions())

8


In [14]:
df_dept = df.repartition(5, "Department")

In [15]:
df_small = df_repartitioned.coalesce(2)

print(df_small.rdd.getNumPartitions())

2


[Stage 7:>                                                          (0 + 4) / 4]

In [16]:
df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/employees_csv")

In [17]:
df.filter(df.Salary > 80000).show()

+----------+-----------+-----------+------+---+
|EmployeeID|       Name| Department|Salary|Age|
+----------+-----------+-----------+------+---+
|         3| Employee_3|         HR| 90231| 41|
|         4| Employee_4|         HR| 95529| 26|
|         5| Employee_5|         HR| 99654| 30|
|         6| Employee_6|    Finance| 84902| 43|
|         8| Employee_8|Engineering| 99773| 30|
|        12|Employee_12|  Marketing| 90634| 39|
|        16|Employee_16|  Marketing| 96940| 28|
|        17|Employee_17|Engineering| 95902| 34|
|        20|Employee_20|         HR| 97188| 22|
|        22|Employee_22|      Sales| 88723| 43|
|        26|Employee_26|    Finance| 89529| 26|
|        27|Employee_27|Engineering| 88338| 41|
|        29|Employee_29|         HR| 84905| 37|
|        34|Employee_34|         HR| 90215| 24|
|        37|Employee_37|  Marketing| 99974| 35|
|        39|Employee_39|         HR| 94544| 33|
|        40|Employee_40|  Marketing| 90106| 38|
|        42|Employee_42|    Finance| 823

In [18]:
df.filter(df.Department == "Engineering").show()

+----------+-----------+-----------+------+---+
|EmployeeID|       Name| Department|Salary|Age|
+----------+-----------+-----------+------+---+
|         8| Employee_8|Engineering| 99773| 30|
|        17|Employee_17|Engineering| 95902| 34|
|        19|Employee_19|Engineering| 50186| 26|
|        23|Employee_23|Engineering| 68074| 37|
|        27|Employee_27|Engineering| 88338| 41|
|        28|Employee_28|Engineering| 65132| 38|
|        33|Employee_33|Engineering| 64677| 22|
|        55|Employee_55|Engineering| 59539| 32|
|        59|Employee_59|Engineering| 77321| 28|
|        60|Employee_60|Engineering| 79089| 34|
|        61|Employee_61|Engineering| 83384| 42|
|        62|Employee_62|Engineering| 64634| 38|
|        63|Employee_63|Engineering| 68493| 42|
|        64|Employee_64|Engineering| 51407| 42|
|        65|Employee_65|Engineering| 81825| 26|
|        67|Employee_67|Engineering| 81605| 39|
|        68|Employee_68|Engineering| 91008| 40|
|        70|Employee_70|Engineering| 884

In [19]:
df.filter(df.Age>30).show()

+----------+-----------+-----------+------+---+
|EmployeeID|       Name| Department|Salary|Age|
+----------+-----------+-----------+------+---+
|         1| Employee_1|      Sales| 61114| 34|
|         2| Employee_2|         HR| 57506| 42|
|         3| Employee_3|         HR| 90231| 41|
|         6| Employee_6|    Finance| 84902| 43|
|         7| Employee_7|         HR| 76794| 39|
|         9| Employee_9|      Sales| 71572| 34|
|        10|Employee_10|    Finance| 67648| 36|
|        11|Employee_11|         HR| 51242| 32|
|        12|Employee_12|  Marketing| 90634| 39|
|        13|Employee_13|         HR| 53252| 38|
|        14|Employee_14|      Sales| 71868| 31|
|        15|Employee_15|    Finance| 50784| 43|
|        17|Employee_17|Engineering| 95902| 34|
|        21|Employee_21|      Sales| 55470| 39|
|        22|Employee_22|      Sales| 88723| 43|
|        23|Employee_23|Engineering| 68074| 37|
|        25|Employee_25|  Marketing| 64810| 39|
|        27|Employee_27|Engineering| 883

In [21]:
df.filter(
    (df.Age>30) &
    (df.Salary>80000)).show()

+----------+-----------+-----------+------+---+
|EmployeeID|       Name| Department|Salary|Age|
+----------+-----------+-----------+------+---+
|         3| Employee_3|         HR| 90231| 41|
|         6| Employee_6|    Finance| 84902| 43|
|        12|Employee_12|  Marketing| 90634| 39|
|        17|Employee_17|Engineering| 95902| 34|
|        22|Employee_22|      Sales| 88723| 43|
|        27|Employee_27|Engineering| 88338| 41|
|        29|Employee_29|         HR| 84905| 37|
|        37|Employee_37|  Marketing| 99974| 35|
|        39|Employee_39|         HR| 94544| 33|
|        40|Employee_40|  Marketing| 90106| 38|
|        42|Employee_42|    Finance| 82342| 44|
|        45|Employee_45|    Finance| 89688| 41|
|        54|Employee_54|      Sales| 91062| 44|
|        61|Employee_61|Engineering| 83384| 42|
|        67|Employee_67|Engineering| 81605| 39|
|        68|Employee_68|Engineering| 91008| 40|
|        69|Employee_69|         HR| 86820| 35|
|        70|Employee_70|Engineering| 884